In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Alternative: direct upload of zip file
import zipfile
import os

zip_path = "/content/fairytale_corpus.zip"
extract_dir = "/content/files_unzipped"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

print("All files were unzipped to ", extract_dir, ".")

## Create Dataframe


In [ ]:
import os
import pandas as pd

# Base directory in Google Drive
base_path = "/content/drive/MyDrive/KI_Maerchen/Korpus"

rows = []

for root, dirs, files in os.walk(base_path):
    for file in files:
        if not file.endswith(".txt"):
            continue

        path = os.path.join(root, file)

        # Folder structure:
        # .../Korpus/Andersen/Original/FILE.txt
        parts = root.split(os.sep)

        # Author (Andersen / Grimm)
        author = parts[-2] if len(parts) >= 2 else "Unknown"

        # Type (Original / Copilot / Wikipedia)
        text_type = parts[-1] if len(parts) >= 1 else "Other"

        # Parse filename
        # Format: Year_OriginalAuthor_Source_Author_Title.txt
        name = file.replace(".txt", "")
        parts_file = name.split("_")

        year = parts_file[0] if parts_file[0].isdigit() else None
        original_author = parts_file[1] if len(parts_file) > 1 else None
        source = parts_file[2] if len(parts_file) > 2 else None
        author2 = parts_file[3] if len(parts_file) > 3 else None

        # Title = everything from index 4 onward
        title = " ".join(parts_file[4:]) if len(parts_file) > 4 else name

        # Load text
        with open(path, "r", encoding="utf-8") as f:
            text = f.read()

        rows.append({
            "Author": author,
            "Type": text_type,
            "Year": year,
            "OriginalAuthor": original_author,
            "Source": source,
            "Author2": author2,
            "Title": title,
            "Text": text,
            "Filename": file,
            "Path": path
        })

df = pd.DataFrame(rows)
df.to_csv("fairy_tale_corpus_complete.csv", index=False, encoding="utf-8-sig")
df.head()


In [ ]:
df_grimm = df[df["OriginalAuthor"] == "Grimm"]
df_andersen = df[df["OriginalAuthor"] == "Andersen"]

df_grimm.to_csv("fairytale_corpus_grimm.csv", index=False, encoding="utf-8-sig")
df_andersen.to_csv("fairytale_corpus_andersen.csv", index=False, encoding="utf-8-sig")

In [ ]:
df_grimm.head()


In [ ]:
df_andersen.head()


In [ ]:
df

In [ ]:
#df = pd.read_csv("fairytale_corpus_andersen.csv", sep=",")
#print(df.columns)
#df.head()

In [ ]:
!pip install bertopic sentence-transformers scikit-learn pandas


In [ ]:
import re
import pandas as pd
from pathlib import Path

from bertopic import BERTopic
from sentence_transformers import SentenceTransformer

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction import text


### Topic Modeling Grimm + Andersen

In [ ]:
def clean_text(text: str) -> str:
    # Remove non-breaking spaces and line breaks
    text = text.replace("\xa0", " ").replace("\n", " ")

    # Lowercase
    text = text.lower()

    # Remove text in brackets (e.g. "(1989 film)")
    text = re.sub(r"\([^)]*\)", "", text)

    # Keep letters only (English + German)
    #text = re.sub(r"[^a-zäöüß\s]", " ", text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text


In [ ]:
def split_into_chunks(text, chunk_size=100): # chunk_size=200
    words = text.split()
    return [
        " ".join(words[i:i + chunk_size])
        for i in range(0, len(words), chunk_size)
    ]


In [ ]:
documents = []
sources = []

for index, row in df.iterrows():
    name = row["Title"]
    raw_text = row["Text"]
    cleaned = clean_text(raw_text)
    chunks = split_into_chunks(cleaned, chunk_size=200)

    documents.extend(chunks)
    sources.extend([name] * len(chunks))

### Stop word removal with Spacy

In [ ]:
!python -m spacy download de_core_news_sm

In [ ]:
# Import Package
import spacy
# Load pipeline Pipeline
nlp = spacy.load("de_core_news_sm")
#nlp_english = spacy.load("en_core_web_sm")
stopwords_german = nlp.Defaults.stop_words
#stopwords_english = nlp_english.Defaults.stop_words
print(stopwords_german)
#print(stopwords_english)


In [ ]:
vectorizer_model = CountVectorizer(
    stop_words=list(stopwords_german),
    ngram_range=(1, 2),
    min_df=3
)

### Perform topic modeling

In [ ]:
# Create embedding model
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")


In [ ]:
#Initialize and run BERTopic
topic_model = BERTopic(
    nr_topics=20,
    embedding_model=embedding_model,
    vectorizer_model=vectorizer_model,
    language="german", #multilingual
    min_topic_size=3, #min_topic_size=10, (to increase clusters)
    verbose=True
)

topics, probs = topic_model.fit_transform(documents)

In [ ]:
# Inspect topics
topic_model.get_topic_info()

In [ ]:
topic_info_df = topic_model.get_topic_info()
topic_info_df.to_csv('topic_info.csv', index=False)
print("Topic information saved to topic_info.csv")

In [ ]:
topic_model.get_topic(0)

### Create Wordclouds of topics

In [ ]:
import matplotlib.pyplot as plt
from wordcloud import WordCloud

In [ ]:
# Get all topic information
all_topics_info = topic_model.get_topic_info()

# Filter out topic -1 (outliers) if you don't want a word cloud for it
# You can modify this loop to exclude any specific topics or include only a subset
for index, row in all_topics_info.iterrows():
    topic_id = row['Topic']
    topic_name = row['Name']

    # Get the words and their probabilities for the current topic
    topic_words_scores = topic_model.get_topic(topic_id)

    # Extract words and scores
    word_scores = {word: score for word, score in topic_words_scores}

    # Create the WordCloud object
    wordcloud = WordCloud(width=800, height=400, background_color='white').generate_from_frequencies(word_scores)

    # Display the generated image:
    plt.figure(figsize=(10, 5))
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.axis('off')
    plt.title(f'Word Cloud for Topic {topic_id}: {topic_name}')
    plt.show()

In [ ]:
# Create results dataframe
# To fix the ValueError, we reconstruct the metadata lists (author, type, year)
# by repeating the values for each chunk generated from the original stories.

authors, types, years = [], [], []

for _, row in df.iterrows():
    # Re-calculate how many chunks this specific row produced
    # using the same logic from your 'split_into_chunks' step
    cleaned = clean_text(row["Text"])
    chunks_count = len(split_into_chunks(cleaned, chunk_size=200))

    # Repeat the metadata values for each of those chunks
    authors.extend([row["Author"]] * chunks_count)
    types.extend([row["Type"]] * chunks_count)
    years.extend([row["Year"]] * chunks_count)

# Now all lists have the same length and can be combined into a DataFrame
df_topic_modeling = pd.DataFrame({
    "text": documents,
    "topic": topics,
    "source": sources,
    "author": authors,
    "type": types,
    "year": years
})

df_topic_modeling.head()

In [ ]:
df_topic_modeling.to_csv('df_topic_modeling', index=False)
print("Topic information saved to df_topic_modeling")

### Topic Modeling Andersen

In [ ]:
def clean_text(text: str) -> str:
    # Remove non-breaking spaces and line breaks
    text = text.replace("\xa0", " ").replace("\n", " ")

    # Lowercase
    text = text.lower()

    # Remove text in brackets (e.g. "(1989 film)")
    text = re.sub(r"\([^)]*\)", "", text)

    # Keep letters only (English + German)
    #text = re.sub(r"[^a-zäöüß\s]", " ", text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text


In [ ]:
def split_into_chunks(text, chunk_size=100): # chunk_size=200
    words = text.split()
    return [
        " ".join(words[i:i + chunk_size])
        for i in range(0, len(words), chunk_size)
    ]


In [ ]:
documents = []
sources = []

for index, row in df_andersen.iterrows():
    name = row["Title"]
    raw_text = row["Text"]
    cleaned = clean_text(raw_text)
    chunks = split_into_chunks(cleaned, chunk_size=200)

    documents.extend(chunks)
    sources.extend([name] * len(chunks))

### Stop word removal with Spacy

In [ ]:
vectorizer_model = CountVectorizer(
    stop_words=list(stopwords_german),
    ngram_range=(1, 2),
    min_df=3
)

### Perform topic modeling

In [ ]:
# Create embedding model
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")


In [ ]:
#Initialize and run BERTopic
topic_model_andersen = BERTopic(
    nr_topics=20,
    embedding_model=embedding_model,
    vectorizer_model=vectorizer_model,
    language="german", #multilingual
    min_topic_size=3, #min_topic_size=10, (to increase clusters)
    verbose=True
)

# Corrected: Fit the 'topic_model_andersen' variable
topics, probs = topic_model_andersen.fit_transform(documents)

In [ ]:
# Inspect topics
topic_model_andersen.get_topic_info()

In [ ]:
topic_info_df_andersen = topic_model_andersen.get_topic_info()
topic_info_df_andersen.to_csv('topic_info_andersen.csv', index=False)
print("Topic information saved to topic_info_andersen.csv")

In [ ]:
topic_model_andersen.get_topic(0)

### Create Wordclouds of topics

In [ ]:
# Get all topic information
all_topics_info_andersen = topic_model_andersen.get_topic_info()

# Filter out topic -1 (outliers) if you don't want a word cloud for it
# You can modify this loop to exclude any specific topics or include only a subset
for index, row in all_topics_info_andersen.iterrows():
    topic_id = row['Topic']
    topic_name = row['Name']

    # Get the words and their probabilities for the current topic
    topic_words_scores_andersen = topic_model_andersen.get_topic(topic_id)

    # Extract words and scores
    word_scores_andersen = {word: score for word, score in topic_words_scores_andersen}

    # Create the WordCloud object
    wordcloud_andersen = WordCloud(width=800, height=400, background_color='white').generate_from_frequencies(word_scores_andersen)

    # Display the generated image:
    plt.figure(figsize=(10, 5))
    plt.imshow(wordcloud_andersen, interpolation='bilinear')
    plt.axis('off')
    plt.title(f'Andersen Word Cloud for Topic {topic_id}: {topic_name}')
    plt.show()

In [ ]:
# Create results dataframe
# To fix the ValueError, we reconstruct the metadata lists (author, type, year)
# by repeating the values for each chunk generated from the original stories.

authors, types, years = [], [], []

for _, row in df_andersen.iterrows():
    # Re-calculate how many chunks this specific row produced
    # using the same logic from your 'split_into_chunks' step
    cleaned = clean_text(row["Text"])
    chunks_count = len(split_into_chunks(cleaned, chunk_size=200))

    # Repeat the metadata values for each of those chunks
    authors.extend([row["Author"]] * chunks_count)
    types.extend([row["Type"]] * chunks_count)
    years.extend([row["Year"]] * chunks_count)

# Now all lists have the same length and can be combined into a DataFrame
df_topic_modeling_andersen = pd.DataFrame({
    "text": documents,
    "topic": topics,
    "source": sources,
    "author": authors,
    "type": types,
    "year": years
})

df_topic_modeling_andersen.head()

In [ ]:
df_topic_modeling_andersen.to_csv('df_topic_modelin_andersen', index=False)
print("Topic information saved to df_topic_modeling_andersen")

### Topic Modeling Grimm

In [ ]:
def clean_text(text: str) -> str:
    # Remove non-breaking spaces and line breaks
    text = text.replace("\xa0", " ").replace("\n", " ")

    # Lowercase
    text = text.lower()

    # Remove text in brackets (e.g. "(1989 film)")
    text = re.sub(r"\([^)]*\)", "", text)

    # Keep letters only (English + German)
    #text = re.sub(r"[^a-zäöüß\s]", " ", text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text


In [ ]:
def split_into_chunks(text, chunk_size=100): # chunk_size=200
    words = text.split()
    return [
        " ".join(words[i:i + chunk_size])
        for i in range(0, len(words), chunk_size)
    ]


In [ ]:
documents = []
sources = []

for index, row in df_grimm.iterrows():
    name = row["Title"]
    raw_text = row["Text"]
    cleaned = clean_text(raw_text)
    chunks = split_into_chunks(cleaned, chunk_size=200)

    documents.extend(chunks)
    sources.extend([name] * len(chunks))

### Stop word removal with Spacy

In [ ]:
vectorizer_model = CountVectorizer(
    stop_words=list(stopwords_german),
    ngram_range=(1, 2),
    min_df=3
)

### Perform topic modeling

In [ ]:
# Create embedding model
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")


In [ ]:
#Initialize and run BERTopic
topic_model_grimm = BERTopic(
    nr_topics=20,
    embedding_model=embedding_model,
    vectorizer_model=vectorizer_model,
    language="german", #multilingual
    min_topic_size=3, #min_topic_size=10, (to increase clusters)
    verbose=True
)

# Fit the 'topic_model_grimm' variable
topics, probs = topic_model_grimm.fit_transform(documents)

In [ ]:
# Inspect topics
topic_model_grimm.get_topic_info()

In [ ]:
topic_info_df_grimm = topic_model_grimm.get_topic_info()
topic_info_df_grimm.to_csv('topic_info_grimm.csv', index=False)
print("Topic information saved to topic_info_grimm.csv")

In [ ]:
topic_model_grimm.get_topic(0)

### Create Wordclouds of topics

In [ ]:
# Get all topic information
all_topics_info_grimm = topic_model_grimm.get_topic_info()

# Filter out topic -1 (outliers) if you don't want a word cloud for it
# You can modify this loop to exclude any specific topics or include only a subset
for index, row in all_topics_info_grimm.iterrows():
    topic_id = row['Topic']
    topic_name = row['Name']

    # Get the words and their probabilities for the current topic
    topic_words_scores_grimm = topic_model_grimm.get_topic(topic_id)

    # Extract words and scores
    word_scores_grimm = {word: score for word, score in topic_words_scores_grimm}

    # Create the WordCloud object
    wordcloud_grimm = WordCloud(width=800, height=400, background_color='white').generate_from_frequencies(word_scores_andersen)

    # Display the generated image:
    plt.figure(figsize=(10, 5))
    plt.imshow(wordcloud_grimm, interpolation='bilinear')
    plt.axis('off')
    plt.title(f'Grimm Word Cloud for Topic {topic_id}: {topic_name}')
    plt.show()

In [ ]:
# Create results dataframe
# To fix the ValueError, we reconstruct the metadata lists (author, type, year)
# by repeating the values for each chunk generated from the original stories.

authors, types, years = [], [], []

for _, row in df_grimm.iterrows():
    # Re-calculate how many chunks this specific row produced
    # using the same logic from your 'split_into_chunks' step
    cleaned = clean_text(row["Text"])
    chunks_count = len(split_into_chunks(cleaned, chunk_size=200))

    # Repeat the metadata values for each of those chunks
    authors.extend([row["Author"]] * chunks_count)
    types.extend([row["Type"]] * chunks_count)
    years.extend([row["Year"]] * chunks_count)

# Now all lists have the same length and can be combined into a DataFrame
df_topic_modeling_grimm = pd.DataFrame({
    "text": documents,
    "topic": topics,
    "source": sources,
    "author": authors,
    "type": types,
    "year": years
})

df_topic_modeling_grimm.head()

In [ ]:
df_topic_modeling_grimm.to_csv('df_topic_modelin_grimm', index=False)
print("Topic information saved to df_topic_modeling_grimm")

### compare classic vs wikipedia vs copilot

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Map the 'type' column to the specific groups requested
# 'Original' becomes 'classic', others stay as is (lowercased)
def map_groups(tale_type):
  t = str(tale_type).lower()
  if t == 'original':
    return 'original'
  elif t == 'wikipedia': # Added this condition
    return 'wikipedia'
  elif t == 'copilot': # Added this condition
    return 'copilot'
  return None # Handle other cases, if any

df_topic_modeling['comparison_group'] = df_topic_modeling['type'].apply(map_groups)

# 2. Create the cross-tabulation (Group vs Topic)
topic_counts = pd.crosstab(df_topic_modeling['comparison_group'], df_topic_modeling['topic'])

# 3. Normalize the distribution to get percentages within each group
topic_dist_norm = topic_counts.div(topic_counts.sum(axis=1), axis=0)

# 4. Filter for only the three groups of interest
groups_to_show = ['original', 'wikipedia', 'copilot']
# Ensure we only include groups that exist in the dataframe
available_groups = [g for g in groups_to_show if g in topic_dist_norm.index]
topic_dist_final = topic_dist_norm.loc[available_groups]

# 5. Display the numerical results
print("Topic Distribution by Group (Normalized proportions):")
display(topic_dist_final)

# 6. Visualize the comparison with a heatmap
plt.figure(figsize=(18, 6))
sns.heatmap(topic_dist_final, annot=True, fmt=".2f", cmap="YlGnBu")
plt.title("Comparison of Topic Distributions: original vs Wikipedia vs Copilot")
plt.ylabel("Category")
plt.xlabel("Topic ID")
plt.show()

In [ ]:
topic_dist_final.to_csv("topic_dist_final.csv", index=True)

### compare Andersen classic vs wikipedia vs copilot

In [ ]:
# 1. Standardize the 'type' column to match requested labels exactly
def standardize_andersen_types(tale_type):
    t = str(tale_type).capitalize()
    if t == 'Original': return 'Original'
    if t == 'Wikipedia': return 'Wikipedia'
    if t == 'Copilot': return 'Copilot'
    return t

df_topic_modeling_andersen['comparison_group'] = df_topic_modeling_andersen['type'].apply(standardize_andersen_types)

# 2. Create the cross-tabulation (Group vs Topic)
topic_counts_andersen = pd.crosstab(df_topic_modeling_andersen['comparison_group'], df_topic_modeling_andersen['topic'])

# 3. Normalize the distribution to get percentages within each group
topic_dist_norm_andersen = topic_counts_andersen.div(topic_counts_andersen.sum(axis=1), axis=0)

# 4. Filter for the three groups of interest
groups_to_show_andersen = ['Original', 'Wikipedia', 'Copilot']
available_groups_andersen = [g for g in groups_to_show_andersen if g in topic_dist_norm_andersen.index]
topic_dist_final_andersen = topic_dist_norm_andersen.loc[available_groups_andersen]

# 5. Display the numerical results
print("Andersen Topic Distribution by Group (Normalized proportions):")
display(topic_dist_final_andersen)

# 6. Visualize the comparison with a heatmap
plt.figure(figsize=(18, 6))
sns.heatmap(topic_dist_final_andersen, annot=True, fmt=".2f", cmap="YlGnBu")
plt.title("Andersen Comparison of Topic Distributions: Original vs Wikipedia vs Copilot")
plt.ylabel("Category")
plt.xlabel("Topic ID")
plt.show()

In [ ]:
topic_dist_final_andersen.to_csv("topic_dist_final_andersen.csv", index=True)

### compare Grimm classic vs wikipedia vs copilot

In [ ]:
# 1. Standardize the 'type' column to match requested labels exactly
def standardize_grimm_types(tale_type):
    t = str(tale_type).capitalize()
    if t == 'Original': return 'Original'
    if t == 'Wikipedia': return 'Wikipedia'
    if t == 'Copilot': return 'Copilot'
    return t

df_topic_modeling_grimm['comparison_group'] = df_topic_modeling_grimm['type'].apply(standardize_grimm_types)

# 2. Create the cross-tabulation (Group vs Topic)
topic_counts_grimm = pd.crosstab(df_topic_modeling_grimm['comparison_group'], df_topic_modeling_grimm['topic'])

# 3. Normalize the distribution to get percentages within each group
topic_dist_norm_grimm = topic_counts_grimm.div(topic_counts_grimm.sum(axis=1), axis=0)

# 4. Filter for the three groups of interest
groups_to_show_grimm = ['Original', 'Wikipedia', 'Copilot']
available_groups_grimm = [g for g in groups_to_show_grimm if g in topic_dist_norm_grimm.index]
topic_dist_final_grimm = topic_dist_norm_grimm.loc[available_groups_grimm]

# 5. Display the numerical results
print("Grimm Topic Distribution by Group (Normalized proportions):")
display(topic_dist_final_grimm)

# 6. Visualize the comparison with a heatmap
plt.figure(figsize=(18, 6))
sns.heatmap(topic_dist_final_grimm, annot=True, fmt=".2f", cmap="YlGnBu")
plt.title("Grimm Comparison of Topic Distributions: Original vs Wikipedia vs Copilot")
plt.ylabel("Category")
plt.xlabel("Topic ID")
plt.show()

In [ ]:
topic_dist_final_grimm.to_csv("topic_dist_final_grimm.csv", index=True)

In [ ]:
# Normalized topic distribution by source
topic_distribution_by_source_group = topic_dist_norm.T.sort_values(by="classic", ascending=False)
#topic_distribution_norm.T.sort_values(by="Modern", ascending=False)
topic_distribution_by_source_group



In [ ]:
topic_distribution_by_source_group.to_csv("topic_distribution_by_source_group.csv", index=True)
print("Topic distribution by period saved to topic_distribution_by_source_group.csv")

##Document topic distribution heat map
similar to: [DARIAH-DE/Topics](https://github.com/DARIAH-DE/Topics)

DARIAH unfortunely doenst work

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# 1. Filter for only the original (original) fairy tales
df_original = df_topic_modeling[df_topic_modeling['comparison_group'] == 'original']

# 2. Create the cross-tabulation (Source/Title vs Topic)
tale_topic_counts = pd.crosstab(df_original['source'], df_original['topic'])

# 3. Normalize the distribution
tale_topic_dist = tale_topic_counts.div(tale_topic_counts.sum(axis=1), axis=0)

# 4. Force the table to include all topics from the main model (e.g., -1 to 18)
# This ensures the heatmap shows the full width even if some topics are 0 for original tales
all_possible_topics = sorted(df_topic_modeling['topic'].unique())
tale_topic_dist = tale_topic_dist.reindex(columns=all_possible_topics, fill_value=0)

# 5. Create the heatmap
plt.figure(figsize=(20, 25))
sns.heatmap(tale_topic_dist, annot=False, cmap="YlGnBu", cbar_kws={'label': 'Topic Proportion'})

plt.title("Topic Distribution Across Original Fairy Tales (Full Topic Range)", fontsize=16)
plt.ylabel("Fairy Tale Title", fontsize=12)
plt.xlabel("Topic ID", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# 1. Filter for only the original
df_original = df_topic_modeling[df_topic_modeling['comparison_group'] == 'original']

# 2. Create the cross-tabulation (Source/Title vs Topic)
tale_topic_counts_original = pd.crosstab(df_original['source'], df_original['topic'])

# 3. Normalize the distribution
tale_topic_dist_original = tale_topic_counts_original.div(tale_topic_counts_original.sum(axis=1), axis=0)

# 4. Force the table to include all topics from the main model (e.g., -1 to 18)
# This ensures the heatmap shows the full width even if some topics are 0 for original tales
all_possible_topics = sorted(df_topic_modeling['topic'].unique())
tale_topic_dist_original = tale_topic_dist_original.reindex(columns=all_possible_topics, fill_value=0)

# 5. Create the heatmap
plt.figure(figsize=(20, 25))
sns.heatmap(tale_topic_dist_original, annot=False, cmap="YlOrBr", cbar_kws={'label': 'Topic Proportion'})

plt.title("Topic Distribution Across original Fairy Tales (Full Topic Range)", fontsize=16)
plt.ylabel("Fairy Tale Title", fontsize=12)
plt.xlabel("Topic ID", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# 1. Filter for only the wikipedia
df_wikipedia = df_topic_modeling[df_topic_modeling['comparison_group'] == 'wikipedia']

# 2. Create the cross-tabulation (Source/Title vs Topic)
tale_topic_counts_wiki = pd.crosstab(df_wikipedia['source'], df_wikipedia['topic'])

# 3. Normalize the distribution
tale_topic_dist_wiki = tale_topic_counts_wiki.div(tale_topic_counts_wiki.sum(axis=1), axis=0)

# 4. Force the table to include all topics from the main model (e.g., -1 to 18)
# This ensures the heatmap shows the full width even if some topics are 0 for original tales
all_possible_topics = sorted(df_topic_modeling['topic'].unique())
tale_topic_dist_wiki = tale_topic_dist_wiki.reindex(columns=all_possible_topics, fill_value=0)

# 5. Create the heatmap
plt.figure(figsize=(20, 25))
sns.heatmap(tale_topic_dist_wiki, annot=False, cmap="YlOrBr", cbar_kws={'label': 'Topic Proportion'})

plt.title("Topic Distribution Across Wikipedia Fairy Tales (Full Topic Range)", fontsize=16)
plt.ylabel("Fairy Tale Title", fontsize=12)
plt.xlabel("Topic ID", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:

# 1. Filter for only the copilot
df_copilot = df_topic_modeling[df_topic_modeling['comparison_group'] == 'copilot']

# 2. Create the cross-tabulation (Source/Title vs Topic)
tale_topic_counts_copilot = pd.crosstab(df_copilot['source'], df_copilot['topic'])

# 3. Normalize the distribution
tale_topic_dist_copilot = tale_topic_counts_copilot.div(tale_topic_counts_copilot.sum(axis=1), axis=0)

# 4. Force the table to include all topics from the main model (e.g., -1 to 18)
# This ensures the heatmap shows the full width even if some topics are 0 for original tales
all_possible_topics = sorted(df_topic_modeling['topic'].unique())
tale_topic_dist_copilot = tale_topic_dist_copilot.reindex(columns=all_possible_topics, fill_value=0)

# 5. Create the heatmap
plt.figure(figsize=(20, 25))
sns.heatmap(tale_topic_dist_copilot, annot=False, cmap="Purples", cbar_kws={'label': 'Topic Proportion'})

plt.title("Topic Distribution Across Copilot Fairy Tales (Full Topic Range)", fontsize=16)
plt.ylabel("Fairy Tale Title", fontsize=12)
plt.xlabel("Topic ID", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# The 'trios' list contains fairy tale titles that have 'Original', 'Wikipedia', and 'Copilot' versions.
# It was generated in cell `cfccce01`.

# Filter the topic distributions for only the common fairy tales (trios)
tale_topic_dist_filtered = tale_topic_dist[tale_topic_dist.index.isin(trios)]
tale_topic_dist_wiki_filtered = tale_topic_dist_wiki[tale_topic_dist_wiki.index.isin(trios)]
tale_topic_dist_copilot_filtered = tale_topic_dist_copilot[tale_topic_dist_copilot.index.isin(trios)]

# Ensure all three filtered dataframes have the exact same set of stories and in the same order
# This is crucial for direct visual comparison across subplots.
common_stories = list(set(tale_topic_dist_filtered.index) & \
                      set(tale_topic_dist_wiki_filtered.index) & \
                      set(tale_topic_dist_copilot_filtered.index))
common_stories.sort() # Ensure consistent order

tale_topic_dist_filtered = tale_topic_dist_filtered.loc[common_stories].sort_index()
tale_topic_dist_wiki_filtered = tale_topic_dist_wiki_filtered.loc[common_stories].sort_index()
tale_topic_dist_copilot_filtered = tale_topic_dist_copilot_filtered.loc[common_stories].sort_index()


# Create a figure with three subplots, one for each source type
# Adjust figsize based on the number of common stories found
fig, axes = plt.subplots(3, 1, figsize=(20, 0.7 * len(common_stories) * 3)) # Scale height based on number of stories

# Define distinct color maps for each heatmap
cmap_original = "Greens"
cmap_wikipedia = "Blues"
cmap_copilot = "Purples"

# Heatmap for Original Fairy Tales
sns.heatmap(tale_topic_dist_filtered, annot=False, cmap=cmap_original, cbar=True, ax=axes[0],
            cbar_kws={'label': 'Topic Proportion'})
axes[0].set_title("Topic Distribution Across Original Fairy Tales (Common Trios)", fontsize=16)
axes[0].set_ylabel("Fairy Tale Title", fontsize=12)
axes[0].tick_params(axis='x', rotation=0) # Keep x-axis labels horizontal

# Heatmap for Wikipedia Fairy Tales
sns.heatmap(tale_topic_dist_wiki_filtered, annot=False, cmap=cmap_wikipedia, cbar=True, ax=axes[1],
            cbar_kws={'label': 'Topic Proportion'})
axes[1].set_title("Topic Distribution Across Wikipedia Fairy Tales (Common Trios)", fontsize=16)
axes[1].set_ylabel("Fairy Tale Title", fontsize=12)
axes[1].tick_params(axis='x', rotation=0)

# Heatmap for Copilot Fairy Tales
sns.heatmap(tale_topic_dist_copilot_filtered, annot=False, cmap=cmap_copilot, cbar=True, ax=axes[2],
            cbar_kws={'label': 'Topic Proportion'})
axes[2].set_title("Topic Distribution Across Copilot Fairy Tales (Common Trios)", fontsize=16)
axes[2].set_ylabel("Fairy Tale Title", fontsize=12)
axes[2].set_xlabel("Topic ID", fontsize=12)
axes[2].tick_params(axis='x', rotation=0)

plt.tight_layout() # Adjust subplot parameters for a tight layout
plt.show()


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# Ensure common_stories and all_possible_topics are defined
# (These should be available from the previous run of GYHpa3UIgilW and cfccce01)

# Re-filter dataframes for common stories and standardize their column order
# This ensures all dataframes have the same stories in the same order and all topics
tale_topic_dist_filtered = tale_topic_dist.loc[common_stories].reindex(columns=all_possible_topics, fill_value=0)
tale_topic_dist_wiki_filtered = tale_topic_dist_wiki.loc[common_stories].reindex(columns=all_possible_topics, fill_value=0)
tale_topic_dist_copilot_filtered = tale_topic_dist_copilot.loc[common_stories].reindex(columns=all_possible_topics, fill_value=0)

# Create a list to store dataframes for concatenation
combined_df_list = []

# Define colors for each source type for visual distinction in labels/legend
row_color_map = {
    'Original': 'seagreen',
    'Wikipedia': 'cornflowerblue',
    'Copilot': 'mediumorchid'
}

# Prepare data for the combined heatmap
# Each story will have three rows: Original, Wikipedia, Copilot
for story in common_stories:
    # Original
    original_row = tale_topic_dist_filtered.loc[story].to_frame().T
    original_row.index = pd.MultiIndex.from_product([[story], ['Original']])
    combined_df_list.append(original_row)

    # Wikipedia
    wikipedia_row = tale_topic_dist_wiki_filtered.loc[story].to_frame().T
    wikipedia_row.index = pd.MultiIndex.from_product([[story], ['Wikipedia']])
    combined_df_list.append(wikipedia_row)

    # Copilot
    copilot_row = tale_topic_dist_copilot_filtered.loc[story].to_frame().T
    copilot_row.index = pd.MultiIndex.from_product([[story], ['Copilot']])
    combined_df_list.append(copilot_row)

# Concatenate all the story-version rows into a single DataFrame
combined_df = pd.concat(combined_df_list)

# Create the figure and axes for the heatmap
fig, ax = plt.subplots(figsize=(20, len(combined_df) * 0.5)) # Scale height dynamically

# Plot the heatmap
sns.heatmap(combined_df, annot=False, cmap='YlGnBu', cbar_kws={'label': 'Topic Proportion'}, ax=ax)

# Customizing Y-axis labels to include the story title and source type, with colored markers
yticklabels_text = []
yticklabels_colors = []
for i, (story, source_type) in enumerate(combined_df.index):
    yticklabels_text.append(f'{chr(0x25CF)} {story} ({source_type})') # Use simple bullet, no mathtext formatting
    yticklabels_colors.append(row_color_map[source_type])

ax.set_yticklabels(yticklabels_text, rotation=0, va='center', fontsize=8)

# Iterate over the tick labels and set their colors individually
for i, tick_label in enumerate(ax.get_yticklabels()):
    tick_label.set_color(yticklabels_colors[i])

# Add a custom legend for the source types with corresponding colors
legend_elements = [Patch(facecolor=color, label=label) for label, color in row_color_map.items()]
ax.legend(handles=legend_elements, title="Source Type", bbox_to_anchor=(1.05, 1), loc='upper left')

ax.set_title("Combined Topic Distribution Across Fairy Tales (Trios)", fontsize=16)
ax.set_ylabel("Fairy Tale Title (Source)", fontsize=12)
ax.set_xlabel("Topic ID", fontsize=12)

# Adjust layout to make room for the legend
plt.tight_layout(rect=[0, 0, 0.88, 1]) # Adjust rect to leave space for legend
plt.show()

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Filter for only the original (classic) fairy tales
df_original = df_topic_modeling[df_topic_modeling['comparison_group'] == 'classic']

# 2. Create the cross-tabulation (Source/Title vs Topic)
tale_topic_counts = pd.crosstab(df_original['source'], df_original['topic'])

# 3. Normalize the distribution to show relative importance of topics per story
tale_topic_dist = tale_topic_counts.div(tale_topic_counts.sum(axis=1), axis=0)

# 4. Create the heatmap
plt.figure(figsize=(20, 25))
sns.heatmap(tale_topic_dist, annot=False, cmap="YlGnBu", cbar_kws={'label': 'Topic Proportion'})

plt.title("Topic Distribution Across Original Fairy Tales", fontsize=16)
plt.ylabel("Fairy Tale Title", fontsize=12)
plt.xlabel("Topic ID", fontsize=12)
plt.tight_layout()
plt.show()

### Trio comparisons
Standardize the story types in the dataset (e.g., to 'Original', 'Wikipedia', 'Copilot'), identify fairy tales that have versions for all three types (forming a "Trio"), and generate a normalized topic distribution heatmap for each identified tale Trio using the data in `df_topic_modeling`.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Standardize story types
def standardize_type(t):
    t = str(t).lower()
    if 'original' in t: return 'Original'
    if 'wikipedia' in t: return 'Wikipedia'
    if 'copilot' in t: return 'Copilot'
    return 'Other'

df_topic_modeling['std_type'] = df_topic_modeling['type'].apply(standardize_type)

# 2. Identify Trios (stories with all 3 versions)
story_counts = df_topic_modeling.groupby('source')['std_type'].nunique()
trios = story_counts[story_counts >= 3].index.tolist()

print(f'Identified {len(trios)} Trios: {trios}')

# 3. Generate a heatmap for each Trio
for story in trios:
    trio_data = df_topic_modeling[df_topic_modeling['source'] == story]
    trio_data = trio_data[trio_data['std_type'].isin(['Original', 'Wikipedia', 'Copilot'])]

    # Cross-tabulate and normalize
    ct = pd.crosstab(trio_data['std_type'], trio_data['topic'])
    ct_norm = ct.div(ct.sum(axis=1), axis=0)

    # Plot
    plt.figure(figsize=(12, 3))
    sns.heatmap(ct_norm, annot=True, cmap='YlGnBu', fmt='.2f')
    plt.title(f'Topic Distribution Comparison: {story}')
    plt.ylabel('Version')
    plt.xlabel('Topic ID')
    plt.show()

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import numpy as np # Import numpy for potential NaN handling

# Ensure common_stories and all_possible_topics are defined
# (These should be available from the previous run of GYHpa3UIgilW and cfccce01)

# Re-filter dataframes for common stories and standardize their column order
# This ensures all dataframes have the same stories in the same order and all topics
tale_topic_dist_filtered = tale_topic_dist.loc[common_stories].reindex(columns=all_possible_topics, fill_value=0)
tale_topic_dist_wiki_filtered = tale_topic_dist_wiki.loc[common_stories].reindex(columns=all_possible_topics, fill_value=0)
tale_topic_dist_copilot_filtered = tale_topic_dist_copilot.loc[common_stories].reindex(columns=all_possible_topics, fill_value=0)

# Create a list to store dataframes for concatenation
combined_df_list = []

# Define colors for each source type for visual distinction in labels/legend
row_color_map = {
    'Original': 'seagreen',
    'Wikipedia': 'cornflowerblue',
    'Copilot': 'mediumorchid'
}

# Prepare data for the combined heatmap
# Each story will have three rows: Original, Wikipedia, Copilot
for story in common_stories:
    # Original
    original_row = tale_topic_dist_filtered.loc[story].to_frame().T
    original_row.index = pd.MultiIndex.from_product([[story], ['Original']])
    combined_df_list.append(original_row)

    # Wikipedia
    wikipedia_row = tale_topic_dist_wiki_filtered.loc[story].to_frame().T
    wikipedia_row.index = pd.MultiIndex.from_product([[story], ['Wikipedia']])
    combined_df_list.append(wikipedia_row)

    # Copilot
    copilot_row = tale_topic_dist_copilot_filtered.loc[story].to_frame().T
    copilot_row.index = pd.MultiIndex.from_product([[story], ['Copilot']])
    combined_df_list.append(copilot_row)

# Concatenate all the story-version rows into a single DataFrame
combined_df = pd.concat(combined_df_list)

# Create the figure and axes for the heatmap
fig, ax = plt.subplots(figsize=(20, len(combined_df) * 0.5)) # Scale height dynamically

# Plot the heatmap
# Set annot=True to show the proportions as text
sns.heatmap(combined_df, annot=True, fmt=".2f", cmap='YlGnBu', cbar_kws={'label': 'Topic Proportion'}, ax=ax)

# Customizing Y-axis labels to include the story title and source type, with colored markers
yticklabels_text = []
yticklabels_colors = []
for i, (story, source_type) in enumerate(combined_df.index):
    yticklabels_text.append(f'{chr(0x25CF)} {story} ({source_type})') # Use simple bullet
    yticklabels_colors.append(row_color_map[source_type])

ax.set_yticklabels(yticklabels_text, rotation=0, va='center', fontsize=8)

# Iterate over the tick labels and set their colors individually
for i, tick_label in enumerate(ax.get_yticklabels()):
    tick_label.set_color(yticklabels_colors[i])

# --- Customizing annotation text color based on source type and hiding 0.00 values ---
# The texts are stored in ax.texts. We need to iterate through them and color them.
# The order of texts corresponds to the data order (row by row, then column by column)
num_rows = combined_df.shape[0]
num_cols = combined_df.shape[1]

for i, text_obj in enumerate(ax.texts):
    row_idx = i // num_cols
    col_idx = i % num_cols

    # Get the original value from the DataFrame
    value = combined_df.iloc[row_idx, col_idx]

    # If the value is 0.0, set the text to an empty string
    if value == 0.0:
        text_obj.set_text("")
    else:
        # Set the color of the annotation text
        # Get the source type for the current row
        _, source_type_for_row = combined_df.index[row_idx]
        text_obj.set_color(row_color_map[source_type_for_row])
        text_obj.set_fontsize(7) # Adjust font size for annotations if needed

# Add a custom legend for the source types with corresponding colors
legend_elements = [Patch(facecolor=color, label=label) for label, color in row_color_map.items()]
ax.legend(handles=legend_elements, title="Source Type", bbox_to_anchor=(1.05, 1), loc='upper left')

ax.set_title("Combined Topic Distribution Across Fairy Tales (Trios)", fontsize=16)
ax.set_ylabel("Fairy Tale Title (Source)", fontsize=12)
ax.set_xlabel("Topic ID", fontsize=12)

# Adjust layout to make room for the legend
plt.tight_layout(rect=[0, 0, 0.88, 1]) # Adjust rect to leave space for legend
plt.show()

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# Ensure common_stories and all_possible_topics are defined
# (These should be available from the previous run of GYHpa3UIgilW and cfccce01)

# Re-filter dataframes for common stories and standardize their column order
# This ensures all dataframes have the same stories in the same order and all topics
tale_topic_dist_filtered = tale_topic_dist.loc[common_stories].reindex(columns=all_possible_topics, fill_value=0)
tale_topic_dist_wiki_filtered = tale_topic_dist_wiki.loc[common_stories].reindex(columns=all_possible_topics, fill_value=0)
tale_topic_dist_copilot_filtered = tale_topic_dist_copilot.loc[common_stories].reindex(columns=all_possible_topics, fill_value=0)

# Create a list to store dataframes for concatenation
combined_df_list = []

# Define colors for each source type for visual distinction in labels/legend
row_color_map = {
    'Original': 'seagreen',
    'Wikipedia': 'cornflowerblue',
    'Copilot': 'mediumorchid'
}

# Prepare data for the combined heatmap
# Each story will have three rows: Original, Wikipedia, Copilot
for story in common_stories:
    # Original
    original_row = tale_topic_dist_filtered.loc[story].to_frame().T
    original_row.index = pd.MultiIndex.from_product([[story], ['Original']])
    combined_df_list.append(original_row)

    # Wikipedia
    wikipedia_row = tale_topic_dist_wiki_filtered.loc[story].to_frame().T
    wikipedia_row.index = pd.MultiIndex.from_product([[story], ['Wikipedia']])
    combined_df_list.append(wikipedia_row)

    # Copilot
    copilot_row = tale_topic_dist_copilot_filtered.loc[story].to_frame().T
    copilot_row.index = pd.MultiIndex.from_product([[story], ['Copilot']])
    combined_df_list.append(copilot_row)

# Concatenate all the story-version rows into a single DataFrame
combined_df = pd.concat(combined_df_list)

# Create the figure and axes for the heatmap
fig, ax = plt.subplots(figsize=(20, len(combined_df) * 0.5)) # Scale height dynamically

# Plot the heatmap
sns.heatmap(combined_df, annot=False, cmap='YlGnBu', cbar_kws={'label': 'Topic Proportion'}, ax=ax)

# Customizing Y-axis labels to include the story title and source type, with colored markers
yticklabels_text = []
yticklabels_colors = []
for i, (story, source_type) in enumerate(combined_df.index):
    yticklabels_text.append(f'{chr(0x25CF)} {story} ({source_type})') # Use simple bullet, no mathtext formatting
    yticklabels_colors.append(row_color_map[source_type])

ax.set_yticklabels(yticklabels_text, rotation=0, va='center', fontsize=8)

# Iterate over the tick labels and set their colors individually
for i, tick_label in enumerate(ax.get_yticklabels()):
    tick_label.set_color(yticklabels_colors[i])

# Add a custom legend for the source types with corresponding colors
legend_elements = [Patch(facecolor=color, label=label) for label, color in row_color_map.items()]
ax.legend(handles=legend_elements, title="Source Type", bbox_to_anchor=(1.05, 1), loc='upper left')

ax.set_title("Combined Topic Distribution Across Fairy Tales (Trios)", fontsize=16)
ax.set_ylabel("Fairy Tale Title (Source)", fontsize=12)
ax.set_xlabel("Topic ID", fontsize=12)

# Adjust layout to make room for the legend
plt.tight_layout(rect=[0, 0, 0.88, 1]) # Adjust rect to leave space for legend
plt.show()